# Edit Search

Run bounded search over direction layers, application spans, and edit strengths on the selection split.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("FRS_REPO_URL", "").strip()
REPO_DIR = Path("/kaggle/working/false-refusal-suppression")

if not REPO_URL:
    raise RuntimeError("Set FRS_REPO_URL in the Kaggle environment before running this notebook.")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[train,dev]"], check=True)

os.chdir(REPO_DIR)
src_path = REPO_DIR / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(REPO_DIR)

In [ ]:
from pathlib import Path

MODEL_ID = "Qwen/Qwen3-4B"
SPLIT_DIR = Path("data/processed/splits/sample_small")
DIRECTION_ARTIFACT = Path("artifacts/directions/qwen3_sample_borderline_vs_easy.json")
SEARCH_ARTIFACT = Path("artifacts/edits/qwen3_sample_search.json")

print(MODEL_ID)
print(SEARCH_ARTIFACT)

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/search_edits.py",
        "--model-id",
        MODEL_ID,
        "--direction-artifact",
        str(DIRECTION_ARTIFACT),
        "--selection-split",
        str(SPLIT_DIR / "selection.jsonl"),
        "--output",
        str(SEARCH_ARTIFACT),
        "--top-k-layers",
        "3",
        "--strength",
        "0.25",
        "--strength",
        "0.5",
        "--strength",
        "1.0",
        "--span-width",
        "1",
        "--span-width",
        "3",
        "--module-type",
        "attn_out",
        "--module-type",
        "mlp_down",
        "--include-norm-preserving",
    ],
    check=True,
 )

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

with open(SEARCH_ARTIFACT, "r", encoding="utf-8") as handle:
    candidates = json.load(handle)

df = pd.DataFrame(candidates)
display(df.head(10))

if not df.empty:
    ax = df.plot.scatter(
        x="false_refusal_rate",
        y="true_refusal_rate",
        s=df["capability_retention"].clip(lower=0.05) * 300,
        figsize=(6, 6),
        title="Edit search candidates",
    )
    ax.set_xlabel("false refusal rate")
    ax.set_ylabel("true refusal rate")
    plt.show()